# KnowWow - Work as usual, knowledge grows

**제조 업무의 반복 패턴과 다른 처리가 발생했을 때만 짧게 질문해 판단 조건을 지식으로 남기는 AI 도우미**

- 핵심 기능: `Case 입력 → 코드 기반 Pattern/Gap 판정 → Prompt → LLM 질문 → 사용자 답변 → Structured Output → 확인 후 저장`
- 데이터: 의도적으로 설계한 Synthetic Comment 24건
- 범위: 온톨로지 구축은 제외하고, 하루 MVP에 맞춰 JSON 데이터와 검증 가능한 Pattern Signature 사용
- 안전 원칙: LLM은 패턴이나 정답을 만들지 않고 **질문 생성·답변 구조화·검색 결과 설명**만 담당

> 루트 `.env`의 실제 API 키로 모든 코드를 다시 실행하고, 질문 생성·답변 구조화 결과를 셀 출력에 저장했습니다. 예시로 사용하는 것은 업무 Comment 데이터와 테스트 답변이며 AI 출력은 실제 모델 결과입니다.


## 1. 기획 의도 — 왜 LLM인가?

제조 Comment에는 처리 결과는 남지만, 담당자가 당시 고려한 숨은 조건은 빠지는 경우가 많습니다. 단순 검색은 유사 사례를 찾을 수 있어도 자유로운 설명에서 새로운 조건을 뽑고, 상황에 맞는 중립적 질문을 만드는 데 한계가 있습니다.

KnowWow는 이미 데이터로 설명되는 정상 처리는 묻지 않습니다. **같은 상황의 다수 처리와 다른 처리가 관찰된 경우에만** LLM이 한 문장의 질문을 만들고, 사용자의 자연어 답변을 구조화합니다.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (candidate / "data/comment_cases.json").is_file()
                     and (candidate / "ai-service").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("KnowWow 프로젝트 폴더 또는 제출파일 폴더에서 실행해주세요.")

sys.path.insert(0, str(PROJECT_ROOT / "ai-service"))
cases = json.loads((PROJECT_ROOT / "data/comment_cases.json").read_text(encoding="utf-8"))
scenarios = json.loads((PROJECT_ROOT / "data/demo_scenarios.json").read_text(encoding="utf-8"))

print("project: KnowWow")
print(f"synthetic cases: {len(cases)}")
print(f"demo scenarios: {len(scenarios)}")


project: KnowWow
synthetic cases: 24
demo scenarios: 3


## 2. Deterministic Context Engineering

LLM에 원시 데이터 전체를 넘기지 않습니다. 먼저 코드가 아래 5개 필드로 같은 상황을 묶고, Action 분포를 계산합니다.

`issue_type + equipment + system + material_status + drawing_status`

지원 사례 3건 이상, 다수 Action 비율 67% 이상일 때만 안정 패턴으로 취급합니다. 패턴과 현재 Action이 다를 때 `ACTION_VARIANT`로 판정하며, 이 판정에는 LLM을 사용하지 않습니다.


In [2]:
SIGNATURE_FIELDS = ("issue_type", "equipment", "system")

def signature(item):
    return (
        *(item[field] for field in SIGNATURE_FIELDS),
        item["context"]["material_status"],
        item["context"]["drawing_status"],
    )

grouped = defaultdict(list)
for item in cases:
    grouped[signature(item)].append(item)

patterns = []
for index, (_, members) in enumerate(
    sorted(grouped.items(), key=lambda pair: min(item["case_id"] for item in pair[1])), start=1
):
    if len(members) < 2:
        continue
    distribution = Counter(item["action"] for item in members)
    majority_action, majority_count = sorted(
        distribution.items(), key=lambda pair: (-pair[1], pair[0])
    )[0]
    ratio = majority_count / len(members)
    patterns.append({
        "pattern_id": f"PATTERN-{index:03d}",
        "signature": {
            "issue_type": members[0]["issue_type"],
            "equipment": members[0]["equipment"],
            "system": members[0]["system"],
            "material_status": members[0]["context"]["material_status"],
            "drawing_status": members[0]["context"]["drawing_status"],
        },
        "support_count": len(members),
        "action_distribution": dict(distribution),
        "majority_action": majority_action,
        "majority_ratio": ratio,
        "supporting_case_ids": [item["case_id"] for item in members],
        "stable": len(members) >= 3 and ratio >= 0.67,
    })

print(f"mined patterns: {len(patterns)}")
for pattern in patterns:
    print(pattern["pattern_id"], pattern["support_count"], pattern["action_distribution"], pattern["stable"])


mined patterns: 4
PATTERN-001 8 {'TRANSFER_TO_PRODUCTION': 6, 'SITE_CHECK_THEN_TRANSFER': 1, 'DRAWING_REVISION': 1} True
PATTERN-002 5 {'DRAWING_REVISION': 4, 'REQUEST_CLARIFICATION': 1} True
PATTERN-003 5 {'MATERIAL_REQUEST': 4, 'TRANSFER_TO_PRODUCTION': 1} True
PATTERN-004 6 {'REQUEST_CLARIFICATION': 5, 'DRAWING_REVISION': 1} True


In [3]:
pattern_by_signature = {tuple(pattern["signature"].values()): pattern for pattern in patterns}

def detect_gap(case):
    pattern = pattern_by_signature.get(signature(case))
    if pattern is None:
        return {"status": "NO_PATTERN", "requires_interview": False}
    variant = pattern["stable"] and case["action"] != pattern["majority_action"]
    return {
        "status": "ACTION_VARIANT" if variant else "NONE",
        "requires_interview": variant,
        "current_action": case["action"],
        "majority_action": pattern["majority_action"],
        "pattern_id": pattern["pattern_id"],
    }

for case_id in ("CASE-001", "CASE-008"):
    item = next(item for item in cases if item["case_id"] == case_id)
    print(case_id, detect_gap(item))


CASE-001 {'status': 'NONE', 'requires_interview': False, 'current_action': 'TRANSFER_TO_PRODUCTION', 'majority_action': 'TRANSFER_TO_PRODUCTION', 'pattern_id': 'PATTERN-001'}
CASE-008 {'status': 'ACTION_VARIANT', 'requires_interview': True, 'current_action': 'DRAWING_REVISION', 'majority_action': 'TRANSFER_TO_PRODUCTION', 'pattern_id': 'PATTERN-001'}


## 3. 핵심 LangChain 체인: 입력 → 프롬프트 → LLM → 질문

`ChatPromptTemplate`에는 역할, 금지 조건, 출력 형식을 넣었습니다. `StrOutputParser`는 채팅 메시지 객체에서 질문 문자열만 꺼냅니다. 둘이 없다면 프롬프트 재사용과 한 문장 출력 계약을 코드 밖에서 일관되게 관리하기 어렵습니다.

이 제출 노트북은 실제 API 키가 있어야 실행됩니다. 가짜 질문을 출력하는 대체 모델은 사용하지 않습니다. `ChatPromptTemplate | ChatModel | StrOutputParser`를 실제 모델로 실행한 결과를 아래에 저장했습니다.


In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from app.knowledge_service import micro_question_context
from app.models import MicroQuestionRequest
from app.terminology import ACTION_LABELS, humanize_chat_text, label_of

load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError("실제 질문 생성 결과를 저장하려면 루트 .env에 OPENAI_API_KEY를 입력해주세요.")

micro_question_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 제조 현장의 경험을 기록하도록 돕는 AI입니다.\n"
        "이번 업무의 처리 이유를 추측하지 마십시오.\n"
        "비슷한 과거 업무의 처리와 이번 처리의 차이를 확인하는 질문 한 개만 만드십시오.\n"
        "원인 예시를 제시하거나 답을 유도하지 마십시오.\n"
        "입력에 제공된 쉬운 한국어 표현을 그대로 사용하십시오.\n"
        "영문 코드나 Case, Action, Context, Pattern 같은 시스템 용어는 쓰지 마십시오.\n"
        "과거에 가장 많았던 처리와 이번 처리를 언급하고 어떤 상황이 달랐는지 물으십시오.\n"
        "'도면 개정'은 정확히 '도면 개정'이라고 쓰고, 질문 한 문장만 출력하십시오."
    )),
    ("human", "이번 업무:\n{current_case}\n\n비슷한 과거 업무에서 관찰된 내용:\n{matched_pattern}"),
])

model = init_chat_model(
    model=os.getenv("MODEL_NAME", "gpt-4o-mini"),
    model_provider=os.getenv("MODEL_PROVIDER", "openai"),
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

micro_question_chain = micro_question_prompt | model | StrOutputParser()
print("LIVE_MODE - 실제 모델 호출")
print("chain: ChatPromptTemplate | ChatModel | StrOutputParser")


LIVE_MODE - 실제 모델 호출
chain: ChatPromptTemplate | ChatModel | StrOutputParser


In [5]:
target_case = next(item for item in cases if item["case_id"] == "CASE-008")
target_pattern = pattern_by_signature[signature(target_case)]
request = MicroQuestionRequest.model_validate({
    "current_case": target_case,
    "matched_pattern": target_pattern,
})
current_case_prompt, matched_pattern_prompt = micro_question_context(request)

question = humanize_chat_text(micro_question_chain.invoke({
    "current_case": json.dumps(current_case_prompt, ensure_ascii=False, indent=2),
    "matched_pattern": json.dumps(matched_pattern_prompt, ensure_ascii=False, indent=2),
}).strip())

print("입력 Case:", target_case["case_id"])
print("Gap:", detect_gap(target_case)["status"])
print("LLM 질문:", question)


입력 Case: CASE-008
Gap: ACTION_VARIANT
LLM 질문: 가장 많이 했던 처리인 '생산 부서로 넘겨 처리'와 이번 처리인 '도면 개정'의 상황은 어떤 점에서 달랐나요?


## 4. Structured Output — 답변을 저장 가능한 지식으로

사용자 답변을 바로 DB에 넣지 않고 Pydantic 스키마로 제한합니다. `model.with_structured_output(PersonalKnowledgeExtraction)`으로 실제 모델 결과를 받고, 명확하지 않은 값은 `null`로 남깁니다. 저장 전 화면에서 사용자가 확인합니다.


In [6]:
from app.models import PersonalKnowledgeExtraction
from app.prompts import KNOWLEDGE_EXTRACTION_PROMPT

user_answer = "실제 설치 위치에 다른 장비가 있어서 그대로 설치할 수 없었습니다."

extraction_chain = KNOWLEDGE_EXTRACTION_PROMPT | model.with_structured_output(PersonalKnowledgeExtraction)
extracted = extraction_chain.invoke({
    "current_case": json.dumps(target_case, ensure_ascii=False, indent=2),
    "matched_pattern": json.dumps(target_pattern, ensure_ascii=False, indent=2),
    "question": question,
    "answer": user_answer,
})

print(extracted.model_dump_json(indent=2))
print("저장 정책: 사용자 확인 전에는 Personal Knowledge로 확정하지 않음")


{
  "new_context": {
    "name": "installation_location_issue",
    "value": "EQUIPMENT_CONFLICT"
  },
  "rationale": "설치 위치에 다른 장비가 있어 설치가 불가능했다는 점에서 두 처리 방식이 달랐다.",
  "exception": null
}
저장 정책: 사용자 확인 전에는 Personal Knowledge로 확정하지 않음


## 5. Retriever · Vector Store · Tool-calling Agent

추가 컴포넌트는 기능을 늘리기 위한 장식이 아니라 근거를 통제하기 위해 사용했습니다.

- `Document`: Case / Personal Knowledge / Organization Pattern의 출처 ID와 유형을 함께 보존
- `OpenAIEmbeddings + InMemoryVectorStore`: 표현이 달라도 의미가 비슷한 과거 경험 검색
- `@tool + create_agent`: 사례·확정 지식·패턴 검색을 분리하고, 사용한 Tool과 Evidence ID 반환
- 서버 후처리: 실제 검색된 ID가 아닌 LLM 인용은 제거

아래 색인은 실제 임베딩 API로 24개 예시 업무 문서를 등록합니다. 문서는 예시이지만 임베딩과 검색 구성은 실제 컴포넌트입니다.


In [7]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(
        page_content=(
            f"과거 업무 사례 {item['case_id']}. 이슈 {item['issue_type']}. "
            f"Comment: {item['comment_text']} Action: {item['action']} Outcome: {item['outcome']}."
        ),
        metadata={"doc_type": "CASE", "source_id": item["case_id"]},
    )
    for item in cases
]

embeddings = OpenAIEmbeddings(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    api_key=os.getenv("OPENAI_API_KEY"),
)
case_store = InMemoryVectorStore(embedding=embeddings)
case_store.add_documents(documents)

@tool
def search_similar_cases(query: str, k: int = 5) -> str:
    '''의미가 비슷한 과거 제조 업무 Case와 실제 Source ID를 검색합니다.'''
    found = case_store.similarity_search(query, k=min(k, 5))
    return json.dumps([
        {"source_id": doc.metadata["source_id"], "content": doc.page_content}
        for doc in found
    ], ensure_ascii=False)

print("indexed documents:", len(documents))
print("registered tool:", search_similar_cases.name)
print("production tools: search_similar_cases, search_personal_knowledge, search_org_patterns, get_case_detail")


indexed documents: 24
registered tool: search_similar_cases
production tools: search_similar_cases, search_personal_knowledge, search_org_patterns, get_case_detail


## 6. 입력을 바꾼 테스트 시나리오 비교

대표 사례 외에 처리 방식과 사용자 답변이 다른 2건을 같은 질문·구조화 체인으로 다시 실행합니다. 아래 기대 조건은 평가용 데이터일 뿐 모델에 입력하지 않습니다. 실제 출력과 비교해 잘된 부분과 오차를 확인합니다.


In [8]:
for scenario in scenarios:
    if scenario["case_id"] == target_case["case_id"]:
        continue
    item = next(case for case in cases if case["case_id"] == scenario["case_id"])
    matched = pattern_by_signature[signature(item)]
    assert detect_gap(item)["requires_interview"]
    request = MicroQuestionRequest.model_validate({"current_case": item, "matched_pattern": matched})
    current_prompt, pattern_prompt = micro_question_context(request)
    generated = humanize_chat_text(micro_question_chain.invoke({
        "current_case": json.dumps(current_prompt, ensure_ascii=False, indent=2),
        "matched_pattern": json.dumps(pattern_prompt, ensure_ascii=False, indent=2),
    }).strip())
    structured = extraction_chain.invoke({
        "current_case": json.dumps(item, ensure_ascii=False, indent=2),
        "matched_pattern": json.dumps(matched, ensure_ascii=False, indent=2),
        "question": generated,
        "answer": scenario["answer"],
    })
    print(f"\n[{item['case_id']}]")
    print("이번 처리:", label_of(ACTION_LABELS, item["action"], "기타 처리"))
    print("가장 많았던 처리:", label_of(ACTION_LABELS, matched["majority_action"], "기타 처리"))
    print("AI 질문:", generated)
    print("담당자 답변:", scenario["answer"])
    print("AI가 정리한 근거:", structured.rationale)
    print("추출한 조건:", structured.new_context.model_dump() if structured.new_context else None)
    print("평가용 기대 조건:", scenario["expected_context_name"], scenario["expected_context_value"])



[CASE-018]
이번 처리: 생산 부서로 넘겨 처리
가장 많았던 처리: 필요한 자재 요청
AI 질문: 가장 많이 했던 처리인 '필요한 자재 요청'과 이번에 한 '생산 부서로 넘겨 처리'는 어떤 상황이 달랐나요?
담당자 답변: 동일 사양의 대체 자재가 이미 현장에 확보되어 있었습니다.
AI가 정리한 근거: 대체 자재가 확보되어 있어 처리 방식이 달라졌다.
추출한 조건: {'name': 'alternative_material_available', 'value': 'YES'}
평가용 기대 조건: substitute_material_availability AVAILABLE

[CASE-024]
이번 처리: 도면 개정
가장 많았던 처리: 관련 내용 추가 확인
AI 질문: 가장 많이 했던 처리인 '관련 내용 추가 확인'과 이번 처리인 '도면 개정'의 상황은 어떻게 달랐나요?
담당자 답변: 기본설계 단계에서 선주와 합의한 변경사항을 최종 합의 메일에서 확인했습니다.
AI가 정리한 근거: 기본설계 단계에서 선주와 합의한 변경사항을 확인했다는 점이 이번 처리와 관련된 상황이다.
추출한 조건: None
평가용 기대 조건: owner_approved_change CONFIRMED


### 이번 실행 결과에서 확인한 점

- `CASE-008`: 설치 위치의 장비 간섭을 새 조건으로 추출했습니다.
- `CASE-018`: 대체 자재 확보라는 의미는 추출했지만, 평가용 이름·값 표기와는 다르게 표현했습니다.
- `CASE-024`: 합의 메일 확인을 판단 근거에는 반영했지만 새 조건 필드는 비워 두었습니다. 모호한 표현에서 구조화 누락이 생기는 한계로, 사람의 확인이 필요합니다.

실제 모델 응답은 재실행 시 달라질 수 있으며, 평가용 기대 조건은 프롬프트에 전달하지 않았습니다.


## 7. 구현 범위와 한계

**구현된 범위**

1. 24건 더미 데이터 검증
2. Spring Boot의 결정론적 Pattern Mining / Gap Detection
3. LangChain LCEL 질문 생성, Structured Output, Vector Store, Retriever, Tools, Agent
4. 사용자 확인 후 Personal Knowledge JSON 저장
5. Next.js 대시보드, My Work, Micro-interview, My Knowledge, Team Knowledge, Agent UI

**현재 한계**

- 온톨로지와 지식 그래프는 하루 MVP 범위에서 제외했습니다.
- 패턴 임계값(지원 3건, 67%)은 작은 Synthetic Data에 맞춘 값이라 운영 데이터로 재검증해야 합니다.
- In-memory Vector Store는 재시작하면 재구축되며 대규모 데이터에 적합하지 않습니다.
- 사용자 답변이 짧거나 모호하면 구조화 결과가 `null`이 될 수 있습니다.
- 본 노트북은 실제 모델로 3건을 실행했지만, 작은 예시만으로 질문·추출 품질을 일반화할 수 없습니다.

**다음 개선**: 운영 DB, 영속 Vector DB, 승인 워크플로, 평가 데이터셋, LangSmith trace, 이후 검증된 Context를 기반으로 한 온톨로지 후보 관리.


## 8. 채점 기준 대응표

| 기준 | 노트북/구현 근거 |
|---|---|
| 주제 선정 | 반복 업무 기록에 빠진 판단 조건을 최소 질문으로 수집 |
| 문제 해결 | `CASE-001`과 `CASE-008` 비교, 3개 답변 시나리오 |
| 프롬프트 설계 | 역할·추측 금지·비유도·한 문장 출력 조건 |
| 체인 구성 | `ChatPromptTemplate \| ChatModel \| StrOutputParser` |
| 추가 컴포넌트 | Structured Output, Document, Vector Store, Retriever, Tools, Agent |
| 후처리 | 사용자 확인 후 저장, Evidence ID 검증 |
| 한계/개선 | 임계값·더미 데이터·인메모리 저장소·온톨로지 제외 명시 |

전체 실행 방법과 서비스 구조는 루트 `README.md`, 상세 기획은 같은 `제출파일` 폴더의 `KnowWow_구현_명세서.md`를 참고하세요. 현재 제출용 파일명은 `3반_정다운_KnowWow.ipynb`입니다.
